# Phase 1 — Audio Dataset & Inspection (Google Colab)

Goal: load and inspect the recordings in `speaker_01/`.

Phase 1 runs fine on **CPU**. A GPU (Runtime → Change runtime type → T4 GPU) is only needed in later phases.

## 1. Check PyTorch and GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU - that's fine for Phase 1.")

## 2. Clone the repository and install requirements

In [ ]:
import os
if not os.path.exists("/content/Voice_engine"):
    !git clone https://github.com/8919134556/Voice_engine.git /content/Voice_engine
%cd /content/Voice_engine
!git pull
!pip install -q -r requirements.txt

## 3. Get your private dataset

**Option A (recommended) — Google Drive.** Put your recordings in Drive like this:

```
MyDrive/voice_engine/dataset/speaker_01/audio_001.wav
MyDrive/voice_engine/dataset/speaker_01/audio_002.wav
MyDrive/voice_engine/dataset/speaker_01/audio_003.wav
```

Then run the next cell (it will ask for permission to access Drive).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = "/content/drive/MyDrive/voice_engine/dataset"
!ls -R "{DATASET_DIR}"

**Option B — upload directly** (files are deleted when the Colab session ends).
Skip this cell if you used Option A.

In [ ]:
# Run ONLY if you are not using Google Drive.
import os, shutil
from google.colab import files

DATASET_DIR = "/content/Voice_engine/dataset"
os.makedirs(f"{DATASET_DIR}/speaker_01", exist_ok=True)
uploaded = files.upload()          # choose audio_001.wav, audio_002.wav, audio_003.wav
for name in uploaded:
    shutil.move(name, f"{DATASET_DIR}/speaker_01/{name}")
!ls -l "{DATASET_DIR}/speaker_01"

## 4. Run the Phase 1 inspection script

In [ ]:
!python scripts/inspect_dataset.py --dataset "{DATASET_DIR}"

## 5. Same thing step by step, in Python

This shows what the script does internally: **WAV file → loader → tensor → inspection**.

In [ ]:
from pathlib import Path
from src.audio import list_audio_files, load_audio, preprocess_audio, inspect_audio, format_report

speaker_dir = Path(DATASET_DIR) / "speaker_01"
wav_files = list_audio_files(speaker_dir)
print("Found:", [p.name for p in wav_files])

# Load the first file and look at the raw tensor
waveform, sr = load_audio(wav_files[0])
print("\nShape:", waveform.shape, "| dtype:", waveform.dtype, "| sample rate:", sr)
print("First 10 samples:", waveform[0, :10])

# Convert to the project standard (mono, 16 kHz)
mono16k, sr16k = preprocess_audio(waveform, sr)
print("After preprocessing:", mono16k.shape, sr16k)

In [ ]:
# Full report for every file
for path in wav_files:
    print(format_report(inspect_audio(path)))
    print()

## 6. Listen to a recording (optional)

In [ ]:
from IPython.display import Audio
Audio(mono16k.numpy()[0], rate=sr16k)